# Simple RAG Pipeline

A complete, minimal RAG (Retrieval Augmented Generation) pipeline.

This notebook demonstrates the full end-to-end RAG flow:
1. Load sample documents
2. Chunk the documents into smaller pieces
3. Generate embeddings with OpenAI and store them in a Pinecone vector database
4. Accept a user question
5. Retrieve the most relevant chunks via semantic search
6. Build an augmented prompt that includes the retrieved context
7. Send the prompt to GPT-4o and get a grounded response

**Key difference from ChromaDB:** Pinecone is a managed cloud vector database -- no local storage needed. However, unlike ChromaDB which auto-embeds documents, Pinecone requires you to generate embeddings yourself. We use OpenAI's `text-embedding-3-small` model for this.

**Prerequisites:** Make sure you have the following environment variables set:
- `OPENAI_API_KEY` -- for embeddings and LLM calls
- `PINECONE_API_KEY` -- for the Pinecone vector database

**Note:** This notebook requires the `DOCUMENTS` data from `sample_documents.ipynb`. You can either:
1. Run `sample_documents.ipynb` first and copy the `DOCUMENTS` variable, or
2. The `DOCUMENTS` data is included inline below for convenience.

## Install Dependencies

In [ ]:
!pip install openai pinecone

## Imports

In [ ]:
import os
import textwrap

from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec

## Initialize Clients

Set up the OpenAI client (for embeddings and LLM calls) and the Pinecone client (for vector storage and search).

In [ ]:
# Initialize clients
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

# Embedding configuration
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536

def get_embeddings(texts: list[str]) -> list[list[float]]:
    """Generate embeddings using OpenAI."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    return [item.embedding for item in response.data]

## Load Sample Documents

Import our sample knowledge base. The `DOCUMENTS` list contains policies, FAQs, product descriptions, and company info for a fictional company called Acme Corp.

**Option A:** If you have already run `sample_documents.ipynb`, you can import from there.

**Option B:** The `DOCUMENTS` data is defined inline in the cell below for standalone use.

In [ ]:
# Import our sample knowledge base
# If running standalone, this cell defines DOCUMENTS inline.
# If you prefer, you can instead run sample_documents.ipynb and import from there.

from sample_documents import DOCUMENTS

## Step 1 -- Chunking

LLMs and vector search work best with smaller, focused pieces of text. We split each document into chunks of roughly `chunk_size` characters, with an overlap so we don't accidentally cut an idea in half.

In [ ]:
def chunk_document(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    """
    Split a document into overlapping chunks.

    Args:
        text:       The full document text.
        chunk_size: Target size of each chunk in characters.
        overlap:    Number of overlapping characters between consecutive chunks.

    Returns:
        A list of text chunks.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        # Move the window forward, but keep `overlap` characters of context
        start += chunk_size - overlap
    return chunks

### Prepare Chunks for Pinecone

Take the raw documents and produce three parallel lists:
- `ids` -- A unique ID for each chunk
- `texts` -- The chunk text
- `metadatas` -- Metadata attached to each chunk (source title, category, etc.)

Note: Unlike ChromaDB which stores documents directly, Pinecone stores vectors. We'll store the chunk text inside the metadata so we can retrieve it later.

In [ ]:
def prepare_chunks(documents: list[dict]) -> tuple[list[str], list[str], list[dict]]:
    """
    Take the raw documents and produce three parallel lists:
      - ids:        A unique ID for each chunk
      - texts:      The chunk text
      - metadatas:  Metadata attached to each chunk (source title, category, etc.)
    """
    ids = []
    texts = []
    metadatas = []

    for doc_idx, doc in enumerate(documents):
        chunks = chunk_document(doc["content"])
        for chunk_idx, chunk_text in enumerate(chunks):
            # Create a unique, readable ID for every chunk
            chunk_id = f"doc{doc_idx}_chunk{chunk_idx}"
            ids.append(chunk_id)
            texts.append(chunk_text)
            metadatas.append(
                {
                    "source_title": doc["title"],
                    "category": doc["metadata"]["category"],
                    "chunk_index": chunk_idx,
                }
            )

    return ids, texts, metadatas

## Step 2 -- Build the Vector Store

Pinecone is a managed cloud vector database. Unlike ChromaDB which auto-embeds documents using a built-in model, Pinecone requires you to generate embeddings yourself. This gives you full control over which embedding model to use.

We use OpenAI's `text-embedding-3-small` model to generate embeddings, then upsert the vectors into a Pinecone index.

In [ ]:
def build_vector_store(ids: list[str], texts: list[str], metadatas: list[dict]):
    """Create Pinecone index and load chunks."""
    INDEX_NAME = "acme-knowledge-base"

    # Delete if exists (for clean re-runs)
    if INDEX_NAME in [idx.name for idx in pc.list_indexes()]:
        pc.delete_index(INDEX_NAME)

    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

    import time
    time.sleep(1)

    index = pc.Index(INDEX_NAME)

    # Generate embeddings
    embeddings = get_embeddings(texts)

    # Build vectors (store text in metadata)
    vectors = []
    for doc_id, embedding, text, metadata in zip(ids, embeddings, texts, metadatas):
        meta = {**metadata, "text": text}
        vectors.append({"id": doc_id, "values": embedding, "metadata": meta})

    # Upsert in batches
    batch_size = 100
    for i in range(0, len(vectors), batch_size):
        index.upsert(vectors=vectors[i:i+batch_size])

    print(f"  Loaded {len(vectors)} chunks into the vector store.")
    return index

## Step 3 -- Retrieve Relevant Chunks

Given a user question, we embed it with the same OpenAI model and ask Pinecone to find the most similar chunks via cosine similarity search.

In [ ]:
def retrieve(index, query: str, top_k: int = 5):
    """
    Search the vector store for chunks most relevant to the query.

    Args:
        index:  The Pinecone index to search.
        query:  The user's question in natural language.
        top_k:  How many chunks to retrieve.

    Returns:
        Pinecone query results with matches containing scores and metadata.
    """
    query_embedding = get_embeddings([query])[0]

    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    return results

## Step 4 -- Build the Augmented Prompt

This is the "Augmented" part of RAG. We take the retrieved chunks and inject them into the prompt so the LLM can answer based on real data.

In [ ]:
def build_prompt(query: str, retrieved_docs: list[str], sources: list[str]) -> str:
    """
    Construct the augmented prompt.

    The prompt has three parts:
      1. System instruction -- tells the LLM how to behave
      2. Retrieved context -- the documents we found
      3. The user's original question

    Args:
        query:          The user's question.
        retrieved_docs: List of chunk texts from the vector DB.
        sources:        List of source titles for each chunk.

    Returns:
        The complete prompt string to send to the LLM.
    """
    # Format each retrieved chunk with its source title
    context_sections = []
    for i, (doc_text, source) in enumerate(zip(retrieved_docs, sources), 1):
        context_sections.append(f"[Document {i}: {source}]\n{doc_text}")

    context_block = "\n\n---\n\n".join(context_sections)

    prompt = (
        f"Using ONLY the context provided below, answer the user's question. "
        f"If the context does not contain enough information to answer the "
        f"question, say: 'I don't have enough information to answer that.'\n\n"
        f"CONTEXT:\n\n{context_block}\n\n"
        f"QUESTION: {query}"
    )
    return prompt

## Step 5 -- Generate a Response with GPT-4o

We send the augmented prompt to GPT-4o. Because the prompt contains the retrieved documents, GPT-4o's answer will be grounded in our actual data rather than its general training knowledge.

In [ ]:
def generate_response(prompt: str) -> str:
    """
    Send the augmented prompt to GPT-4o and return the response.
    """
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    # Extract the text from the response
    return response.choices[0].message.content

## Full RAG Pipeline

This function ties everything together:

**Load docs -> Chunk -> Store -> Retrieve -> Augment -> Generate**

In [ ]:
def run_rag_pipeline(question: str) -> str:
    """
    Execute the complete RAG pipeline for a given question.

    This is the function that ties everything together:
      Load docs -> Chunk -> Store -> Retrieve -> Augment -> Generate
    """
    print("=" * 60)
    print("RAG PIPELINE")
    print("=" * 60)

    # --- Phase 1: Indexing (normally done once, offline) ---
    print("\n1. Chunking documents...")
    ids, texts, metadatas = prepare_chunks(DOCUMENTS)
    print(f"  Created {len(ids)} chunks from {len(DOCUMENTS)} documents.")

    print("\n2. Building vector store...")
    index = build_vector_store(ids, texts, metadatas)

    # --- Phase 2: Querying (happens per question) ---
    print(f"\n3. Retrieving relevant chunks for: '{question}'")
    results = retrieve(index, question, top_k=5)

    # Pull out the retrieved documents and their source titles
    retrieved_docs = [match.metadata["text"] for match in results.matches]
    retrieved_metas = [match.metadata for match in results.matches]
    sources = [match.metadata["source_title"] for match in results.matches]

    print(f"  Found {len(retrieved_docs)} relevant chunks:")
    for i, source in enumerate(sources, 1):
        print(f"    {i}. {source}")

    print("\n4. Building augmented prompt...")
    prompt = build_prompt(question, retrieved_docs, sources)

    print("\n5. Generating response with GPT-4o...")
    response = generate_response(prompt)

    return response

## Run the Pipeline

Check for the API keys, then run the RAG pipeline on several example questions that demonstrate RAG's ability to answer from our data.

In [ ]:
# Check for API keys
missing_keys = []
if not os.environ.get("OPENAI_API_KEY"):
    missing_keys.append("OPENAI_API_KEY")
if not os.environ.get("PINECONE_API_KEY"):
    missing_keys.append("PINECONE_API_KEY")

if missing_keys:
    print("ERROR: Please set the following environment variables:")
    for key in missing_keys:
        print(f"  export {key}='your-key-here'")
else:
    print("API keys found. Ready to run the pipeline.")

In [ ]:
# Example questions that demonstrate RAG's ability to answer from our data
example_questions = [
    "What is the return policy for electronics?",
    "How many days of PTO do employees with 4 years of experience get?",
    "What are the specs of the UltraBook 15?",
    "How does the Acme Rewards loyalty program work?",
]

In [ ]:
# Run the pipeline for each question
for question in example_questions:
    response = run_rag_pipeline(question)

    print("\n" + "-" * 60)
    print(f"QUESTION: {question}")
    print("-" * 60)
    # Wrap long lines for readability
    for line in response.split("\n"):
        print(textwrap.fill(line, width=80))
    print("\n")

## Cleanup

Delete the Pinecone index to avoid ongoing costs. In production, you would keep the index running.

In [ ]:
# Clean up - delete the Pinecone index
pc.delete_index("acme-knowledge-base")